# House Prices: BaseMLP repeated-CV submission

## tl;dr

- 모델·feature·keep-all 정책은 고정하고 validation을 5-fold × seeds 42/2026/3407로 변경했다.
- 15-fold 평균 RMSLE는 0.145881 ± 0.026967이며 이전 validation 진단을 정확히 재현했다.
- 각 행의 세 out-of-training-fold 예측을 평균한 OOF ensemble RMSLE는 0.134232다.
- seed 42의 첫 5-model 예측은 기존 BaseMLP 제출과 최대 차이 5.82e-11로 동일하다.
- 15-model 제출은 기존 5-model 제출과 1,459행 모두 다르고 평균 절대 차이는 3,630.97이다.
- 사용자 제공 Kaggle 화면에서 새 CSV의 public RMSLE는 0.12313이며 private은 unverified다.

## Context & Methods

기존 BaseMLP와 비교해 바뀌는 실험 요인은 하나다. 기존 seed 42의 5-fold를
split seed 42, 2026, 3407에서 반복한다. 각 fold는 전처리기와 수동 PyTorch
`BaseMLP`를 새로 적합하고 validation RMSLE 기준 best checkpoint를 복원한다.

- training row policy: `keep_all`
- 입력: 기존 79개 원본 predictor, 생성 feature 0개, `Id`는 모델에서 제외
- 전처리·모델·optimizer·loss·early stopping: 역사적 BaseMLP와 동일
- 역사적 `Id=524, YearRemodAdd=2007` 모델링 복사본 보정도 유지한다. 이를 제거하면
  validation 외 요소까지 달라지므로 이번 **validation-only** 비교에는 포함하지 않는다.
- test 제출: 15개 restored-checkpoint log 예측의 평균을 `expm1`
- 외부 프로젝트 스크립트 import: 0건

추가로 각 학습 행이 매 seed에서 정확히 한 번 validation인 OOF 예측 3개를 평균해,
test ensemble 효과와 같은 방향의 leakage-free OOF ensemble RMSLE도 계산한다.

## Data

`data/train.csv`, `data/test.csv`, `data/sample_submission.csv`를 읽는다.
원본 파일은 수정하지 않으며 제출 열·행 수·dtype·test Id 순서를 실제 sample과 검증한다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
BASELINE_SUBMISSION_PATH = ROOT / "submissions" / "submission_BASEMLP-20260719-2H-01.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "validation_test" /
    "basemlp_repeated_cv_submission.ipynb"
)
REPORT_DIR = ROOT / "reports" / "validation_test"
ARTIFACT_DIR = ROOT / "artifacts" / "validation_test"
FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_fold_results.csv"
OOF_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_oof.csv"
TEST_PREDICTIONS_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_test_predictions.csv"
SUMMARY_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_summary.json"
COMPARISON_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_comparison.json"
REFERENCE_FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_validation_fold_results.csv"
SUBMISSION_PATH = ROOT / "submissions" / "submission_BASEMLP-20260719-REPEATED-CV-3SEED-NB-06.csv"
RUN_ID = "BASEMLP-20260719-REPEATED-CV-3SEED-NB-06"
PUBLIC_PROBE_SCORES = {
    "keep_all": 0.12579,
    "exclude_1299": 0.13016,
    "exclude_524_1299": 0.13426,
}
POLICIES = {
    "keep_all": (),
    "exclude_1299": (1299,),
    "exclude_524_1299": (524, 1299),
}
SPLIT_SEEDS = (42, 2026, 3407)

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"test: {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("external project script imports: 0")

train: 1,460 rows × 81 columns
test: 1,459 rows × 80 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0


## Fixed BaseMLP recipe

아래 행 정리와 fold-fitted median/scaling/one-hot은 기존 BaseMLP와 동일하다.
새 feature나 이상 플래그는 추가하지 않는다.

In [2]:
def clean_rows_historical_basemlp(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned

X = clean_rows_historical_basemlp(
    train.drop(columns="SalePrice"), categorical_columns
)
X_test = clean_rows_historical_basemlp(
    test, categorical_columns
)
assert float(
    X.loc[X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2007
assert list(X_test.columns) == [
    column for column in X.columns
]
print("Historical BaseMLP preprocessing reproduced; the validation/ensemble is the only experiment change.")


def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

Historical BaseMLP preprocessing reproduced; the validation/ensemble is the only experiment change.


## Repeated 5-fold training

세 split seed 각각에서 5-fold를 수행해 총 15개 best checkpoint, 전처리기,
epoch log를 저장한다. 이전 validation 진단의 15개 keep-all fold 점수와도 대조한다.

In [3]:
OUTPUT_DIR = ARTIFACT_DIR / RUN_ID
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
PREPROCESSOR_DIR = OUTPUT_DIR / "preprocessors"
EPOCH_LOG_DIR = OUTPUT_DIR / "logs"
for directory in (CHECKPOINT_DIR, PREPROCESSOR_DIR, EPOCH_LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

oof_log_by_seed = np.full(
    (len(SPLIT_SEEDS), len(train)), np.nan, dtype=np.float64
)
test_log_predictions: list[np.ndarray] = []
test_prediction_labels: list[str] = []
fold_rows: list[dict[str, object]] = []
started = time.perf_counter()

for seed_index, split_seed in enumerate(SPLIT_SEEDS):
    splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=split_seed)
    for fold, (train_index, valid_index) in enumerate(
        splitter.split(X), start=1
    ):
        split_number = seed_index * N_SPLITS + fold
        fold_seed = split_seed + fold
        checkpoint_path = (
            CHECKPOINT_DIR / f"seed_{split_seed}_fold_{fold}_best.pt"
        )
        preprocessor_path = (
            PREPROCESSOR_DIR / f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
        )
        epoch_log_path = (
            EPOCH_LOG_DIR / f"seed_{split_seed}_fold_{fold}_epochs.csv"
        )

        preprocessor = make_preprocessor(list(NUMERIC_COLUMNS))
        X_train = preprocessor.fit_transform(
            X.iloc[train_index]
        ).astype(np.float32)
        X_valid = preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        X_test_fold = preprocessor.transform(X_test).astype(np.float32)
        joblib.dump(preprocessor, preprocessor_path)

        model, training_result = train_fold(
            X_train,
            y_log[train_index],
            X_valid,
            y_log[valid_index],
            seed=fold_seed,
            experiment_id=RUN_ID,
            fold=split_number,
            checkpoint_path=checkpoint_path,
            epoch_log_path=epoch_log_path,
        )
        valid_log_prediction = predict_log_target(
            model,
            X_valid,
            training_result["target_mean"],
            training_result["target_std"],
        )
        test_log_prediction = predict_log_target(
            model,
            X_test_fold,
            training_result["target_mean"],
            training_result["target_std"],
        )
        oof_log_by_seed[seed_index, valid_index] = valid_log_prediction
        test_log_predictions.append(test_log_prediction)
        test_prediction_labels.append(f"seed_{split_seed}_fold_{fold}")
        fold_rows.append({
            "split_seed": split_seed,
            "fold": fold,
            "split_number": split_number,
            "model_seed": fold_seed,
            "training_rows": len(train_index),
            "validation_rows": len(valid_index),
            "encoded_features": X_train.shape[1],
            "best_epoch": training_result["best_epoch"],
            "stopping_epoch": training_result["stopping_epoch"],
            "rmsle": training_result["restored_validation_rmsle"],
            "checkpoint_path": str(checkpoint_path.relative_to(ROOT)),
            "preprocessor_path": str(preprocessor_path.relative_to(ROOT)),
            "epoch_log_path": str(epoch_log_path.relative_to(ROOT)),
        })
        print(
            f"[{split_number:02d}/15] split_seed={split_seed}, fold={fold}, "
            f"RMSLE={training_result['restored_validation_rmsle']:.6f}, "
            f"best_epoch={training_result['best_epoch']}"
        )

duration_seconds = time.perf_counter() - started
fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(FOLD_RESULTS_PATH, index=False)
assert not np.isnan(oof_log_by_seed).any()
assert len(test_log_predictions) == 15

# The earlier validation experiment used the same recipe. Exact score agreement
# proves this notebook changed the repeated split/inference path, not the model.
reference = pd.read_csv(REFERENCE_FOLD_RESULTS_PATH)
reference = reference.loc[
    reference["base_strategy"].eq("repeated_kfold5_3seeds")
    & reference["policy"].eq("keep_all")
].sort_values(["component_seed", "fold"])
current = fold_results.sort_values(["split_seed", "fold"])
assert reference[["component_seed", "fold"]].to_numpy().tolist() == current[
    ["split_seed", "fold"]
].to_numpy().tolist()
assert np.allclose(
    reference["rmsle"].to_numpy(), current["rmsle"].to_numpy(),
    rtol=0, atol=1e-12,
)
print(f"15-fold training completed in {duration_seconds:.2f} seconds")
print("All 15 fold scores exactly reproduce the prior validation diagnostic.")

[01/15] split_seed=42, fold=1, RMSLE=0.140891, best_epoch=12


[02/15] split_seed=42, fold=2, RMSLE=0.131788, best_epoch=43


[03/15] split_seed=42, fold=3, RMSLE=0.222199, best_epoch=5


[04/15] split_seed=42, fold=4, RMSLE=0.134316, best_epoch=21


[05/15] split_seed=42, fold=5, RMSLE=0.122514, best_epoch=14


...

[11/15] split_seed=3407, fold=1, RMSLE=0.134099, best_epoch=7


[12/15] split_seed=3407, fold=2, RMSLE=0.138253, best_epoch=4


[13/15] split_seed=3407, fold=3, RMSLE=0.167069, best_epoch=28


[14/15] split_seed=3407, fold=4, RMSLE=0.148617, best_epoch=51


[15/15] split_seed=3407, fold=5, RMSLE=0.109095, best_epoch=8
15-fold training completed in 8.29 seconds
All 15 fold scores exactly reproduce the prior validation diagnostic.


## Validation results

15개 fold의 평균·표준편차와, 행마다 세 개의 out-of-fold 예측만 평균한
leakage-free 3-seed OOF ensemble RMSLE를 분리해 보고한다.

In [4]:
fold_scores = fold_results["rmsle"].to_numpy(dtype=np.float64)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
seed_oof_scores = {
    str(seed): float(
        np.sqrt(np.mean((y_log - oof_log_by_seed[index]) ** 2))
    )
    for index, seed in enumerate(SPLIT_SEEDS)
}
repeated_ensemble_oof_log = oof_log_by_seed.mean(axis=0)
repeated_ensemble_oof_rmsle = float(
    np.sqrt(np.mean((y_log - repeated_ensemble_oof_log) ** 2))
)

oof_results = pd.DataFrame({
    "Id": train["Id"].to_numpy(dtype=np.int64),
    "target_log1p": y_log,
})
for index, split_seed in enumerate(SPLIT_SEEDS):
    oof_results[f"seed_{split_seed}_oof_log_prediction"] = (
        oof_log_by_seed[index]
    )
oof_results["repeated_ensemble_oof_log_prediction"] = (
    repeated_ensemble_oof_log
)
oof_results["repeated_ensemble_squared_log_error"] = (
    y_log - repeated_ensemble_oof_log
) ** 2
oof_results.to_csv(OOF_RESULTS_PATH, index=False)

assert np.isclose(cv_mean, 0.14588108497354113, rtol=0, atol=1e-12)
assert np.isclose(cv_std, 0.026966873871957453, rtol=0, atol=1e-12)
display(fold_results)
display(pd.DataFrame({
    "metric": [
        "15-fold mean RMSLE",
        "15-fold RMSLE SD",
        "leakage-free 3-seed OOF ensemble RMSLE",
    ],
    "value": [cv_mean, cv_std, repeated_ensemble_oof_rmsle],
}))
print("Per-seed OOF RMSLE:", seed_oof_scores)

,split_seed,fold,split_number,model_seed,training_rows,validation_rows,encoded_features,best_epoch,stopping_epoch,rmsle,checkpoint_path,preprocessor_path,epoch_log_path
0,42,1,1,43,1168,292,327,12,37,0.140891,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
1,42,2,2,44,1168,292,327,43,68,0.131788,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
2,42,3,3,45,1168,292,323,5,30,0.222199,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
3,42,4,4,46,1168,292,324,21,46,0.134316,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
4,42,5,5,47,1168,292,328,14,39,0.122514,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
5,2026,1,6,2027,1168,292,323,7,32,0.141128,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
6,2026,2,7,2028,1168,292,326,30,55,0.179038,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
7,2026,3,8,2029,1168,292,325,5,30,0.128823,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
8,2026,4,9,2030,1168,292,327,11,36,0.149235,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...
9,2026,5,10,2031,1168,292,328,11,36,0.141151,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...,artifacts/validation_test/BASEMLP-20260719-REP...


,metric,value
0,15-fold mean RMSLE,0.145881
1,15-fold RMSLE SD,0.026967
2,leakage-free 3-seed OOF ensemble RMSLE,0.134232


Per-seed OOF RMSLE: {'42': 0.15468769439866675, '2026': 0.14883664497152194, '3407': 0.14071047429531103}


## Submission and comparison

15개 test log 예측을 평균해 제출 CSV를 만든다. seed 42의 첫 다섯 예측만
평균한 결과가 기존 BaseMLP CSV와 정확히 같은지 먼저 확인한 뒤, 15-model
평균과 기존 5-model 평균의 실제 차이를 저장한다.

In [5]:
test_prediction_matrix = np.column_stack(test_log_predictions)
mean_test_log_prediction = test_prediction_matrix.mean(axis=1)
sale_price_prediction = np.expm1(mean_test_log_prediction)

test_prediction_results = pd.DataFrame({
    "Id": test["Id"].to_numpy(dtype=np.int64)
})
for column_index, label in enumerate(test_prediction_labels):
    test_prediction_results[f"{label}_log_prediction"] = (
        test_prediction_matrix[:, column_index]
    )
test_prediction_results["mean_log_prediction"] = mean_test_log_prediction
test_prediction_results["SalePrice"] = sale_price_prediction
test_prediction_results.to_csv(TEST_PREDICTIONS_PATH, index=False)

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
assert sample_submission.columns.tolist() == ["Id", "SalePrice"]
assert len(sample_submission) == len(test)
assert sample_submission["Id"].tolist() == test["Id"].tolist()
submission_frame = sample_submission.copy()
submission_frame["SalePrice"] = sale_price_prediction.astype(np.float64)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_frame.to_csv(SUBMISSION_PATH, index=False)

baseline_submission = pd.read_csv(BASELINE_SUBMISSION_PATH)
assert baseline_submission.columns.tolist() == submission_frame.columns.tolist()
assert baseline_submission["Id"].tolist() == submission_frame["Id"].tolist()
seed42_log_mean = test_prediction_matrix[:, :5].mean(axis=1)
seed42_sale_price = np.expm1(seed42_log_mean)
seed42_max_abs_difference = float(np.max(
    np.abs(seed42_sale_price - baseline_submission["SalePrice"].to_numpy())
))
assert seed42_max_abs_difference <= 1e-9

difference = (
    submission_frame["SalePrice"].to_numpy()
    - baseline_submission["SalePrice"].to_numpy()
)
absolute_difference = np.abs(difference)
relative_difference = absolute_difference / baseline_submission[
    "SalePrice"
].to_numpy()
comparison = {
    "baseline_submission": str(BASELINE_SUBMISSION_PATH.relative_to(ROOT)),
    "repeated_cv_submission": str(SUBMISSION_PATH.relative_to(ROOT)),
    "same_columns": True,
    "same_id_order": True,
    "seed42_five_model_max_abs_difference": seed42_max_abs_difference,
    "different_prediction_rows": int(np.count_nonzero(absolute_difference > 0)),
    "mean_absolute_difference": float(absolute_difference.mean()),
    "median_absolute_difference": float(np.median(absolute_difference)),
    "max_absolute_difference": float(absolute_difference.max()),
    "mean_relative_difference": float(relative_difference.mean()),
    "median_relative_difference": float(np.median(relative_difference)),
    "max_relative_difference": float(relative_difference.max()),
}
COMPARISON_PATH.write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False), encoding="utf-8"
)

submission_validation = {
    "columns": submission_frame.columns.tolist(),
    "rows": int(len(submission_frame)),
    "dtypes": {column: str(dtype) for column, dtype in submission_frame.dtypes.items()},
    "id_matches_test_order": bool(submission_frame["Id"].tolist() == test["Id"].tolist()),
    "unique_ids": bool(submission_frame["Id"].is_unique),
    "missing_predictions": int(submission_frame["SalePrice"].isna().sum()),
    "nonfinite_predictions": int((~np.isfinite(submission_frame["SalePrice"])).sum()),
    "nonpositive_predictions": int((submission_frame["SalePrice"] <= 0).sum()),
    "prediction_min": float(submission_frame["SalePrice"].min()),
    "prediction_max": float(submission_frame["SalePrice"].max()),
    "fold_prediction_count": int(test_prediction_matrix.shape[1]),
    "checkpoint_count": len(list(CHECKPOINT_DIR.glob("*.pt"))),
    "preprocessor_count": len(list(PREPROCESSOR_DIR.glob("*.joblib"))),
    "epoch_log_count": len(list(EPOCH_LOG_DIR.glob("*.csv"))),
}
submission_validation["all_checks_passed"] = bool(
    submission_validation["columns"] == ["Id", "SalePrice"]
    and submission_validation["rows"] == len(test)
    and submission_validation["id_matches_test_order"]
    and submission_validation["unique_ids"]
    and submission_validation["missing_predictions"] == 0
    and submission_validation["nonfinite_predictions"] == 0
    and submission_validation["nonpositive_predictions"] == 0
    and submission_validation["fold_prediction_count"] == 15
    and submission_validation["checkpoint_count"] == 15
    and submission_validation["preprocessor_count"] == 15
    and submission_validation["epoch_log_count"] == 15
)
assert submission_validation["all_checks_passed"]

run_timestamp = datetime.now(timezone.utc).isoformat()
summary = {
    "run_timestamp_utc": run_timestamp,
    "experiment_id": RUN_ID,
    "change_scope": {
        "validation": "KFold(5, shuffle=True) repeated at split seeds 42, 2026, 3407",
        "training_policy": "keep_all",
        "test_ensemble": "mean of 15 restored-checkpoint log predictions, then expm1",
        "model_changes": 0,
        "feature_changes": 0,
        "row_policy_changes": 0,
        "external_project_script_imports": 0,
        "historical_id_524_correction_preserved": True,
    },
    "source": {
        "train_sha256": sha256(TRAIN_PATH),
        "test_sha256": sha256(TEST_PATH),
        "sample_submission_sha256": sha256(SAMPLE_SUBMISSION_PATH),
    },
    "validation": {
        "fold_scores": fold_scores.tolist(),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "seed_oof_rmsle": seed_oof_scores,
        "repeated_ensemble_oof_rmsle": repeated_ensemble_oof_rmsle,
        "reference_15_fold_scores_exactly_reproduced": True,
    },
    "model": {
        "class": "BaseMLP(nn.Module)",
        "architecture": "input->[128,64]->1",
        "optimizer": "Adam(lr=0.001, weight_decay=0)",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "device": str(DEVICE),
    },
    "results": {
        "duration_seconds": duration_seconds,
        "best_epochs": fold_results["best_epoch"].astype(int).tolist(),
        "stopping_epochs": fold_results["stopping_epoch"].astype(int).tolist(),
    },
    "submission_validation": submission_validation,
    "comparison_to_original_five_fold_submission": comparison,
    "kaggle_public_score": "unverified",
    "kaggle_private_score": "unverified",
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "fold_results": str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        "oof_predictions": str(OOF_RESULTS_PATH.relative_to(ROOT)),
        "test_predictions": str(TEST_PREDICTIONS_PATH.relative_to(ROOT)),
        "submission": str(SUBMISSION_PATH.relative_to(ROOT)),
        "checkpoint_dir": str(CHECKPOINT_DIR.relative_to(ROOT)),
        "preprocessor_dir": str(PREPROCESSOR_DIR.relative_to(ROOT)),
        "epoch_log_dir": str(EPOCH_LOG_DIR.relative_to(ROOT)),
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

append_experiment({
    "experiment_id": RUN_ID,
    "datetime": run_timestamp,
    "objective": "Change only BaseMLP validation to repeated 5-fold and create a 15-checkpoint mean-log submission",
    "data_version": f"train sha256={sha256(TRAIN_PATH)}; test sha256={sha256(TEST_PATH)}",
    "split_strategy": "Repeated KFold: KFold(5, shuffle=True) at split seeds 42|2026|3407",
    "folds": "5x3",
    "seed": "42|2026|3407; model seed=split_seed+fold",
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "Historical BaseMLP preprocessing fixed; fold median/scaling/one-hot; Id=524 YearRemodAdd=2007 correction preserved for exact validation-only comparison",
    "features": "79 original predictors; Id excluded; no generated features",
    "model": "Manual PyTorch BaseMLP",
    "architecture": "input->128(ReLU)->64(ReLU)->1; Kaiming hidden/Xavier output; no dropout/batchnorm",
    "optimizer": "Adam(weight_decay=0)",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "best_epoch": json.dumps(fold_results["best_epoch"].astype(int).tolist()),
    "hyperparameters": "keep_all; min_delta=1e-5; test=mean of 15 restored-checkpoint log predictions then expm1; float32; CPU",
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "holdout_score": repeated_ensemble_oof_rmsle,
    "checkpoint_path": str(CHECKPOINT_DIR.relative_to(ROOT)),
    "artifact_path": " | ".join([
        str(SUBMISSION_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
        str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        str(OOF_RESULTS_PATH.relative_to(ROOT)),
        str(TEST_PREDICTIONS_PATH.relative_to(ROOT)),
    ]),
    "result": "15-fold repeated-CV submission created; all submission and exact-reproduction checks passed",
    "interpretation": f"Validation/model-selection is more stable across seeds; leakage-free 3-seed OOF ensemble RMSLE={repeated_ensemble_oof_rmsle:.6f}. New test CSV differs only because ten additional fold models are averaged.",
    "next_action": "Optionally submit this exact CSV once and record public/private separately; do not repeatedly tune to public leaderboard",
})

display(submission_frame.head())
display(pd.DataFrame([comparison]))
print(f"Submission: {SUBMISSION_PATH.relative_to(ROOT)}")
print(f"SHA-256: {sha256(SUBMISSION_PATH)}")
print("All submission checks passed.")

,Id,SalePrice
0,1461,121762.643060
1,1462,160192.617219
2,1463,179977.650306
3,1464,195592.028369
4,1465,187427.242140


,baseline_submission,repeated_cv_submission,same_columns,same_id_order,seed42_five_model_max_abs_difference,different_prediction_rows,mean_absolute_difference,median_absolute_difference,max_absolute_difference,mean_relative_difference,median_relative_difference,max_relative_difference
0,submissions/submission_BASEMLP-20260719-2H-01.csv,submissions/submission_BASEMLP-20260719-REPEAT...,True,True,5.820766e-11,1459,3630.96578,2630.064358,39999.578032,0.02037,0.016588,0.140143


Submission: submissions/submission_BASEMLP-20260719-REPEATED-CV-3SEED-NB-06.csv
SHA-256: 1a428fbc740608a482a24acd1b01f2b8b2d065640d533c62d7ff28c2f1e8b53d
All submission checks passed.


## Takeaways

- 생성 feature, 모델, 전처리, optimizer, loss, 학습 행 정책 변경은 0건이다.
- validation 반복으로 fold 표준편차는 기존 seed-42 단독 0.040707에서 15-fold 0.026967로 낮아졌다.
- 세 seed에서 각 행이 학습에 들어가지 않은 예측만 평균한 OOF ensemble RMSLE 0.134232는 seed 반복의 분산 감소 효과가 크다는 근거다. 다만 test는 15개 모델을 평균하므로 OOF의 3-model 평균과 완전히 같은 추정량은 아니다.
- 새 제출은 1,459행, Id/SalePrice 순서, test Id 순서, 유일 Id, 유한·양수 예측과 15개 checkpoint/preprocessor/log 검사를 모두 통과했다.
- 사용자 제공 Kaggle 화면에서 public RMSLE는 0.12313으로 확인됐고 private은 unverified다.